In [9]:
import pandas as pd
import wn
import algo.no_cover
import algo.costs
import algo.utils
import tqdm
import pprint
import numpy as np
import tabulate
from functools import partial

In [26]:
def format_experiment(name: str) -> str:
    map = {
        '_leafs': 'vanilla',
        '_ml': 'hcp-ref{eq:crc}',
        '_csp4': 'CSP $M=4$',
        '_csp8': 'CSP $M=8$',
        '_layer': 'hcp-ref{eq:hcp_score} layer',
        '_first': 'hcp-ref{eq:hcp_score}',
        '_nd': '   w/o dynamic alpha',
        '_nobc': '  w/o corrected alpha',
    }
    for k, v in map.items():
        if name.endswith(k):
            return v
    return k

def format_dataset(name: str) -> str:
    map = {
        'imagenet': 'img',
        'gtzan': 'gtz',
        'dbpedia': 'dbp',
    }
    for k, v in map.items():
        if name.startswith(k):
            return v
    return k

DATASET_ORDER = [
    'dbpedia',
    'imagenet',
    'gtzan',
]        

ENDING_ORDER = [
    '_leafs',
    '_csp4',
    '_csp8',
    '_layer',
    '_first',
    '_nd',
    '_nobc',
]

def order_dict(
    input_dict, 
    primary=DATASET_ORDER, 
    secondary=ENDING_ORDER,
):
    """
    Orders dictionary items by two-level suffix ordering.

    Args:
        input_dict (dict): Dictionary with string keys.
        primary_suffix_order (list): Primary list of suffixes (first-level sorting).
        secondary_suffix_order (list): Secondary list of suffixes (second-level sorting), optional.

    Returns:
        dict: New dictionary ordered by suffixes.
    """

    def get_suffix_index(key, suffix_list):
        for i, suffix in enumerate(suffix_list):
            if key.endswith(suffix):
                return i
        return len(suffix_list)  # unmatched suffixes get lowest priority

    def get_prefix_index(key, prefix_list):
        for i, prefix in enumerate(prefix_list):
            if key.startswith(prefix):
                return i
        return len(prefix_list)  # unmatched suffixes get lowest priority

    def sort_key(kv):
        key = kv[0]
        primary_index = get_prefix_index(key, primary)
        secondary_index = (
            get_suffix_index(key, secondary)
            if secondary else 0
        )
        return (primary_index, secondary_index, key)

    sorted_items = sorted(input_dict.items(), key=sort_key)
    return dict(sorted_items)

def format_results(results):
    table = []
    for ds, result in order_dict(results).items():
        row = []
        if ds.endswith(ENDING_ORDER[0]):
            row.append(format_dataset(ds))
        else:
            row.append("")
        row.append(format_experiment(ds))
        row += [
            f"{result['covered']:.4f}".lstrip('0'),
            f"pm {result['covered_std']:.2f}".lstrip('0'),
            f"{result['df']['costs'].mean():.2f}".lstrip('0'),
            f"pm {result['df']['costs'].std():.2f}",
            f"{result['ps_lens'].mean():.2f}",
            f"pm {result['ps_lens'].std():.2f}",
            f"{result['n_covered'].mean():.2f}",
            f"pm {result['n_covered'].std():.2f}",
        ]
        if ('lca' in result['df'].columns):
            table.append(row)
            row = [
                '',
                '     +lca',
                f"{result['covered']:.4f}".lstrip('0'),
                f"pm {result['covered_std']:.2f}".lstrip('0'),
                f"{result['df']['lca_costs'].mean():.2f}",
                "pm"+f"{result['df']['lca_costs'].std():.2f}".strip('0'),
                1,
                f"pm 0.00",
                f"{result['df']['lca_n_covered'].mean():.2f}",
                f"pm {result['df']['lca_n_covered'].std():.2f}",
            ]
        table.append(row)
    headers = [
        'dataset',
        'method',
        'coverage',
        'costs',
        'PS size',
        '# covered'
    ]
    return tabulate.tabulate(table, headers=headers, tablefmt="latex_booktabs")


In [51]:
beta_gtz = .25
beta_dbp = .19
beta_img = .04
datasets = [
    # ("gtzan_leafs", "data/gtzan/hcp_0.1_leafs.csv", beta_gtz),
    # ("gtzan_ml", "data/gtzan/hcp_0.1_0.25_ml.csv",beta_gtz),
    ("gtzan_first", "data/gtzan/hcp_0.1_0.25_first.csv",beta_gtz),
    ("gtzan_layer", "data/gtzan/hcp_0.1_0.25_first_layer.csv",beta_gtz),
    # ("gtzan_nd", "data/gtzan/hcp_0.1_0.25_first_nd.csv", beta_gtz),
    # ("gtzan_nobc", "data/gtzan/hcp_0.1_0.25_nobc.csv", beta_gtz),
    # ("gtzan_csp4", "data/gtzan/csp_4.csv", beta_gtz),
    # ("gtzan_csp8", "data/gtzan/csp_8.csv", beta_gtz),

    # ("dbpedia14_leafs", "data/dbpedia14/hcp_0.0003_leafs.csv", beta_dbp),
    # ("dbpedia14_ml", "data/dbpedia14/hcp_0.0003_0.19_ml.csv", beta_dbp),
    ("dbpedia14_first", "data/dbpedia14/hcp_0.0003_0.19_first.csv", beta_dbp),
    ("dbpedia14_layer", "data/dbpedia14/hcp_0.0003_0.19_first_layer.csv", beta_dbp),
    # ("dbpedia14_nd", "data/dbpedia14/hcp_0.0003_0.19_first_nd.csv", beta_dbp),
    # ("dbpedia14_nobc", "data/dbpedia14/hcp_0.0003_0.19_nobc.csv", beta_dbp),
    # ("dbpedia14_csp4", "data/dbpedia14/csp_4.csv", beta_dbp),
    # ("dbpedia14_csp8", "data/dbpedia14/csp_8.csv", beta_dbp),
    # ("imagenet_leafs", "data/imagenet/hcp_0.02_leafs.csv", beta_img),
    # ("imagenet_ml", "data/imagenet/hcp_0.02_0.04_ml.csv", beta_img),
    # ("imagenet_layer", "data/imagenet/hcp_0.02_0.04_first.csv", beta_img),
    # ("imagenet_nd", "data/imagenet/hcp_0.02_0.04_first_nd.csv", beta_img),
    # ("imagenet_nobc", "data/imagenet/hcp_0.02_0.04_nobc.csv", beta_img),
    # ("imagenet_csp4", "data/imagenet/csp_.02_4.csv", beta_img),
    # ("imagenet_csp8", "data/imagenet/csp_.02_8.csv", beta_img),
    # ("imagenet_l", "data/imagenet/hcp_0.04_0.1_last.csv", beta_img),
    # ("imagenet_leafs", "data/imagenet/hcp_0.1_leafs.csv", beta_img),
    # ("imagenet_ml", "data/imagenet/hcp_0.1_0.04_ml.csv", beta_img),
    # ("imagenet_layer", "data/imagenet/hcp_0.1_0.04_first.csv", beta_img),
    # ("imagenet_nd", "data/imagenet/hcp_0.1_0.04_first_nd.csv", beta_img),
    # ("imagenet_nobc", "data/imagenet/hcp_0.1_0.04_nobc.csv", beta_img),
    # ("imagenet_csp4", "data/imagenet/csp_.1_4.csv", beta_img),
    # ("imagenet_csp8", "data/imagenet/csp_.1_8.csv", beta_img),
]

def to_synset(input):
    """Returns the synset 'xxx' for input string 'Synset(xxx)'"""
    idx = input[8:-2]
    synset = wn.synset(idx)
    return synset

def to_synset_scp(input, wordnet=None):
    synsets = wordnet.synsets(input[1:-1])
    if len(synsets) == 0:
        raise ValueError(f"No synset found for {input} in {wordnet}")
    else:
        return synsets[0]
    
def to_synset_scp_img(input, wordnet=None):
    return wordnet.synset(input[1:-1])

def to_depth(input):
    ss = to_synset(input)
    depth = algo.utils.max_synset_depth(ss)
    return depth

def to_human_readable(input):
    if 'subset-' in input:
        input = input.replace('subset-', '')
    ss = to_synset(input)
    return ss.lemmas()[0]

In [12]:
import os.path
for _, f, _ in datasets:
    if not os.path.isfile(f):
        print(f)

data/imagenet/hcp_0.02_0.04_first.csv


In [13]:
results = {}
for name, dataset, beta in datasets:
    print(f"Dataset: {name}")
    if "dbpedia" in name:
        wordnet = wn.Wordnet("dbpedia14-en")
    elif "imagenet" in name:
        wordnet = wn.Wordnet("omw-subset-en")
    elif "gtzan" in name:
        wordnet = wn.Wordnet("gtzan-en")
    else:
        raise ValueError(f"No mapping to wordnet for {name}")
    if "csp" in name:
        to_synset_func = partial(to_synset_scp, wordnet=wordnet)
        if 'imagenet' in name:
            to_synset_func = partial(to_synset_scp_img, wordnet=wn.Wordnet("omw-subset-en"))
        df = pd.read_csv(dataset)
        df['prediction_set_ss'] = df['test_pred_set'].map(
            lambda x: frozenset(map(to_synset_func, x[1:-1].split(", ")))
        )
        df['costs'] = df['prediction_set_ss'].map(
            lambda x:
                algo.costs.cost(
                    beta,
                    x,
                    leaves,
                    None,
                    cost_function_empty = 0.0
            )[0]
        )
        results[name] = {
            'covered': df.test_coverage.mean(),
            'covered_std': df.test_coverage.std(),
            'covered_sem': df.test_coverage.sem(),
            'alpha': 1- df.epsilon.mean(),
            'bf_alpha': None,
            'beta': None,
            'ps_len_mean': df.test_len.mean(),
            'ps_lens': df.test_len,
            # 'label_depth': df.label_depth,
            # 'label_dists': depths - df.label_depth,
            # 'depths': depths,
            'n_covered': df.test_size,
            'best_cover_lens': None,
            'df': df,
            }
    else:
        if name in ('gtzan_leafs', 'dbpedia14_leafs', 'imagenet_leafs', 'imagenet_05_leafs'):
            get_lca = True
        else:
            get_lca = False
        leaves = algo.utils.all_leaves(wordnet)
        df = pd.read_csv(dataset)
        df['covered'] = False
        df['prediction_set_ss'] = None
        depths = pd.Series(dtype=object)
        best_cover_lens = []
        df['prediction_set_ss'] = df['prediction_set'].map(
            lambda x: frozenset(map(to_synset, x[1:-1].split(", ")))
        )
        df['label_ss'] = df['label'].map(to_synset)
        df['covered'] = df.apply(
            lambda x: algo.no_cover.covers_all_nodes(x['prediction_set_ss'], frozenset({x['label_ss']})),
            axis=1
        )
        df['n_covered'] = df.apply(
            lambda x: len(algo.no_cover.covered_nodes(x['prediction_set_ss'], leaves)),
            axis=1
        )
        df['costs'] = df['prediction_set_ss'].map(
            lambda x:
                algo.costs.cost(
                    beta,
                    x,
                    leaves,
                    None,
                    cost_function_empty = 0.0
            )[0]
        )
        if get_lca:
            df['lca'] = df['prediction_set_ss'].map(algo.utils.get_lca)
            df['lca_covered'] = df.apply(lambda x: algo.no_cover.covers_all_nodes(frozenset({x['lca']}), frozenset({x['label_ss']})), axis=1)
            df['lca_n_covered'] = df['lca'].map(lambda x: len(algo.no_cover.covered_nodes(frozenset({x}), leaves)))
            df['lca_costs'] = df.apply(lambda x: 1 + beta * x['lca_n_covered'], axis=1)
        results[name] = {
            'covered': df.covered.mean(),
            'covered_std': df.covered.std(),
            'covered_sem': df.covered.sem(),
            'alpha': df.alpha.mean(),
            'bf_alpha': df.bf_alpha.mean(),
            'beta': df.beta.mean(),
            'ps_len_mean': df.prediction_set_len.mean(),
            'ps_lens': df.prediction_set_len,
            # 'label_depth': df.label_depth,
            # 'label_dists': depths - df.label_depth,
            # 'depths': depths,
            'n_covered': df.n_covered,
            'best_cover_lens': df['best_cover'].apply(len),
            'df': df,
            }

Dataset: gtzan_first
Dataset: gtzan_layer
Dataset: dbpedia14_first
Dataset: dbpedia14_layer
Dataset: imagenet_layer


FileNotFoundError: [Errno 2] No such file or directory: 'data/imagenet/hcp_0.02_0.04_first.csv'

In [14]:
print(format_results(results))

\begin{tabular}{llrlrlrlrl}
\toprule
    &                             &        &         &   dataset & method   &   coverage & costs   &   PS size & \# covered   \\
\midrule
    & hcp-ref\{eq:hcp\_score\} layer & 0.9999 & pm 0.01 &      2.11 & pm 1.05  &       1.29 & pm 0.63 &      4.36 & pm 4.71     \\
    & hcp-ref\{eq:hcp\_score\}       & 0.9999 & pm 0.01 &      2.07 & pm 1.00  &       1.25 & pm 0.55 &      4.3  & pm 4.65     \\
    & hcp-ref\{eq:hcp\_score\} layer & 0.9613 & pm 0.19 &      2.73 & pm 0.97  &       1.19 & pm 0.39 &      6.17 & pm 4.14     \\
    & hcp-ref\{eq:hcp\_score\}       & 0.9637 & pm 0.19 &      2.76 & pm 0.98  &       1.11 & pm 0.32 &      6.57 & pm 4.24     \\
\bottomrule
\end{tabular}


In [15]:
for ds, result in results.items():
    print(ds, result['covered'], result['covered_std'], result['ps_lens'].mean(), result['ps_lens'].std(), result['n_covered'].mean(), result['n_covered'].std())
    print(result['ps_lens'].mean()- 2* result['ps_lens'].sem(), result['ps_lens'].mean()+ 2* result['ps_lens'].sem(), result['n_covered'].mean()- 2* result['n_covered'].std(), result['n_covered'].mean()+ 2* result['n_covered'].sem())
    print(result['df']['costs'].mean(), result['df']['costs'].std())
    print(result['df']['costs'].mean()- 2* df['costs'].sem(),result['df']['costs'].mean()+2* df['costs'].sem())
    if ('lca' in result['df'].columns):
        print(ds + '_lca', result['df'].lca_covered.mean(), result['df'].lca_covered.std(), 1, 0.0, result['df'].lca_n_covered.mean(), result['df'].lca_n_covered.std())
        print(result['df'].lca_n_covered.mean()- 2* result['df'].lca_n_covered.sem(), result['df'].lca_n_covered.mean()+ 2* result['df'].lca_n_covered.sem())

        print(result['df']['lca_costs'].mean(), result['df']['lca_costs'].std())
        print(result['df']['lca_costs'].mean()- 2* result['df']['lca_costs'].sem(), result['df']['lca_costs'].mean()+ 2* result['df']['lca_costs'].sem())

gtzan_first 0.96375 0.18702850605615445 1.1125 0.3161782897173852 6.5725 4.23641094821462
1.0901428187276871 1.134857181272313 -1.90032189642924 6.8720594909375485
2.755625 0.9819074738660678
2.7467792396025326 2.764470760397468
gtzan_layer 0.96125 0.19311928003283488 1.18625 0.3895518268244006 6.17375 4.142464357262352
1.1587045261628859 1.2137954738371142 -2.111178714524703 6.466666463784378
2.7296875 0.969969885778882
2.720841739602532 2.7385332603974675
dbpedia14_first 0.9998571428571429 0.011951539038776656 1.2522321428571428 0.55059260500391 4.299446428571429 4.649426231889056
1.2475787860297316 1.256885499684554 -4.9994060352066825 4.338741252190703
2.0691269642857146 1.0031506396677867
2.060281203888247 2.0779727246831823
dbpedia14_layer 0.9998571428571429 0.011951539038776656 1.2854464285714287 0.6284410100937546 4.357375 4.705507783302796
1.2801351326361625 1.2907577245066948 -5.0536405666055915 4.397143799237165
2.113347678571429 1.0466444850720422
2.1045019181739613 2.12219

In [8]:
for ds, _, _ in datasets:
    print(ds, results[ds]['df']['prediction_set_len'].min())

gtzan_leafs 1


In [ ]:
# datasets = [('imagenet', "data/imagenet/hcp_0.02_betas_cost.csv"),]
datasets = [('imagenet', 'data/imagenet/hcp_0.02_betas_first.csv', beta_img)]
for name, dataset, beta in datasets:
    if "dbpedia" in name:
        wordnet = wn.Wordnet("dbpedia14-en")
    elif "imagenet" in name:
        wordnet = wn.Wordnet("omw-subset-en")
    elif "gtzan" in name:
        wordnet = wn.Wordnet("gtzan-en")
    else:
        raise ValueError(f"No mapping to wordnet for {name}")
    print(f"Dataset: {name}")
    leaves = algo.utils.all_leaves(wordnet)
    df = pd.read_csv(dataset)
    df['covered'] = False
    depths = pd.Series(dtype=object)
    best_cover_lens = []
    df = pd.read_csv(dataset)
    df['covered'] = False
    df['prediction_set_ss'] = None
    depths = pd.Series(dtype=object)
    best_cover_lens = []
    df['prediction_set_ss'] = df['prediction_set'].map(
        lambda x: frozenset(map(to_synset, x[1:-1].split(", ")))
    )
    df['label_ss'] = df['label'].map(to_synset)
    df['covered'] = df.apply(
        lambda x: algo.no_cover.covers_all_nodes(x['prediction_set_ss'], frozenset({x['label_ss']})),
        axis=1
    )
    df['n_covered'] = df.apply(
        lambda x: len(algo.no_cover.covered_nodes(x['prediction_set_ss'], leaves)),
        axis=1
    )
    df['costs'] = df['prediction_set_ss'].map(
        lambda x:
            algo.costs.cost(
                beta,
                x,
                leaves,
                None,
                cost_function_empty = 0.0
        )[0]
    )
    # results[name] = {
    #     'covered': df.covered.mean(),
    #     'covered_std': df.covered.std(),
    #     'covered_sem': df.covered.sem(),
    #     'alpha': df.alpha.mean(),
    #     'bf_alpha': df.bf_alpha.mean(),
    #     'beta': df.beta.mean(),
    #     'ps_len_mean': df.prediction_set_len.mean(),
    #     'ps_lens': df.prediction_set_len,
    #     'label_depth': df.label_depth,
    #     'label_dists': depths - df.label_depth,
    #     'depths': depths,
    #     'n_covered': df.n_covered,
    #     'best_cover_lens': df['best_cover'].apply(len),
    #     }

Dataset: imagenet


In [ ]:
df.groupby('beta')['covered'].mean()

beta
0.00    0.991850
0.01    0.989550
0.02    0.987450
0.03    0.985925
0.04    0.984450
0.05    0.983600
0.06    0.982775
0.07    0.982250
0.08    0.981675
0.09    0.981300
0.10    0.981050
0.20    0.979800
0.30    0.979575
0.40    0.979475
0.50    0.979550
0.60    0.979475
0.70    0.979475
0.80    0.979500
0.90    0.979500
1.00    0.979450
Name: covered, dtype: float64

In [ ]:
df.groupby('beta')['prediction_set_len'].mean()

beta
0.00     0.999825
0.01     1.563900
0.02     2.450625
0.03     3.390975
0.04     4.407700
0.05     5.312825
0.06     6.075425
0.07     6.774025
0.08     7.377450
0.09     7.924900
0.10     8.343450
0.20    10.392475
0.30    10.661050
0.40    10.726000
0.50    10.779100
0.60    10.798025
0.70    10.811450
0.80    10.842000
0.90    10.850175
1.00    10.890100
Name: prediction_set_len, dtype: float64

In [ ]:
df.groupby('beta')['n_covered'].mean()

beta
0.00    334.267250
0.01    218.598050
0.02    158.970575
0.03    121.336725
0.04     92.108900
0.05     71.901825
0.06     58.008475
0.07     47.233100
0.08     39.150750
0.09     32.683925
0.10     28.269575
0.20     12.655650
0.30     11.517900
0.40     11.331400
0.50     11.219800
0.60     11.186800
0.70     11.166500
0.80     11.125650
0.90     11.116275
1.00     11.076275
Name: n_covered, dtype: float64

In [ ]:
datasets = [('imagenet', "data/imagenet/hcp_0.02_betas_first.csv", beta_img),]
# datasets = [('imagenet', 'data/imagenet/hcp_0.02_betas_cost_low.csv', beta_img)]
for name, dataset, beta in datasets:
    if "dbpedia" in name:
        wordnet = wn.Wordnet("dbpedia14-en")
    elif "imagenet" in name:
        wordnet = wn.Wordnet("omw-subset-en")
    elif "gtzan" in name:
        wordnet = wn.Wordnet("gtzan-en")
    else:
        raise ValueError(f"No mapping to wordnet for {name}")
    print(f"Dataset: {name}")
    leaves = algo.utils.all_leaves(wordnet)
    df = pd.read_csv(dataset)
    df['covered'] = False
    depths = pd.Series(dtype=object)
    best_cover_lens = []
    df = pd.read_csv(dataset)
    df['covered'] = False
    df['prediction_set_ss'] = None
    depths = pd.Series(dtype=object)
    best_cover_lens = []
    df['prediction_set_ss'] = df['prediction_set'].map(
        lambda x: frozenset(map(to_synset, x[1:-1].split(", ")))
    )
    df['label_ss'] = df['label'].map(to_synset)
    df['covered'] = df.apply(
        lambda x: algo.no_cover.covers_all_nodes(x['prediction_set_ss'], frozenset({x['label_ss']})),
        axis=1
    )
    df['n_covered'] = df.apply(
        lambda x: len(algo.no_cover.covered_nodes(x['prediction_set_ss'], leaves)),
        axis=1
    )
    df['costs'] = df['prediction_set_ss'].map(
        lambda x:
            algo.costs.cost(
                beta,
                x,
                leaves,
                None,
                cost_function_empty = 0.0
        )[0]
    )

Dataset: imagenet


In [ ]:
df.groupby('beta')['covered'].mean()

beta
0.0    0.991850
0.1    0.981050
0.2    0.979800
0.3    0.979575
0.4    0.979475
0.5    0.979550
0.6    0.979475
0.7    0.979475
0.8    0.979500
0.9    0.979500
1.0    0.979450
Name: covered, dtype: float64

In [ ]:
df.groupby('beta')['prediction_set_len'].mean()

beta
0.0     0.999825
0.1     8.343450
0.2    10.392425
0.3    10.661000
0.4    10.725975
0.5    10.779100
0.6    10.798025
0.7    10.811450
0.8    10.842000
0.9    10.850175
1.0    10.890100
Name: prediction_set_len, dtype: float64

In [ ]:
df.groupby('beta')['n_covered'].mean()

beta
0.0    334.276050
0.1     28.269275
0.2     12.655775
0.3     11.518025
0.4     11.331450
0.5     11.219800
0.6     11.186800
0.7     11.166500
0.8     11.125650
0.9     11.116275
1.0     11.076275
Name: n_covered, dtype: float64

In [ ]:
datasets = [('gtzan', "data/gtzan/hcp_0.1_betas_first.csv", beta_gtz)]
for name, dataset, beta in datasets:
    if "dbpedia" in name:
        wordnet = wn.Wordnet("dbpedia14-en")
    elif "imagenet" in name:
        wordnet = wn.Wordnet("omw-subset-en")
    elif "gtzan" in name:
        wordnet = wn.Wordnet("gtzan-en")
    else:
        raise ValueError(f"No mapping to wordnet for {name}")
    print(f"Dataset: {name}")
    leaves = algo.utils.all_leaves(wordnet)
    df = pd.read_csv(dataset)
    df['covered'] = False
    depths = pd.Series(dtype=object)
    best_cover_lens = []
    df = pd.read_csv(dataset)
    df['covered'] = False
    df['prediction_set_ss'] = None
    depths = pd.Series(dtype=object)
    best_cover_lens = []
    df['prediction_set_ss'] = df['prediction_set'].map(
        lambda x: frozenset(map(to_synset, x[1:-1].split(", ")))
    )
    df['label_ss'] = df['label'].map(to_synset)
    df['covered'] = df.apply(
        lambda x: algo.no_cover.covers_all_nodes(x['prediction_set_ss'], frozenset({x['label_ss']})),
        axis=1
    )
    df['n_covered'] = df.apply(
        lambda x: len(algo.no_cover.covered_nodes(x['prediction_set_ss'], leaves)),
        axis=1
    )
    df['costs'] = df['prediction_set_ss'].map(
        lambda x:
            algo.costs.cost(
                beta,
                x,
                leaves,
                None,
                cost_function_empty = 0.0
        )[0]
    )

Dataset: gtzan


In [ ]:
df.groupby('beta')['covered'].mean()

beta
0.0    0.98125
0.1    0.98125
0.2    0.96375
0.3    0.94250
0.4    0.94250
0.5    0.93000
0.6    0.93000
0.7    0.93000
0.8    0.92375
0.9    0.92375
1.0    0.92375
Name: covered, dtype: float64

In [ ]:
df.groupby('beta')['prediction_set_len'].mean()

beta
0.0    1.0000
0.1    1.0000
0.2    1.1125
0.3    1.4350
0.4    1.4350
0.5    1.9450
0.6    1.9450
0.7    1.9450
0.8    2.4850
0.9    2.4850
1.0    2.4975
Name: prediction_set_len, dtype: float64

In [ ]:
df.groupby('beta')['n_covered'].mean()

beta
0.0    7.47250
0.1    7.47250
0.2    6.57250
0.3    5.44375
0.4    5.44375
0.5    4.42375
0.6    4.42375
0.7    4.42375
0.8    3.74875
0.9    3.74875
1.0    3.73625
Name: n_covered, dtype: float64

In [ ]:
datasets = [('dbpedia', "data/dbpedia14/hcp_0.0003_betas_first.csv", beta_dbp),]
for name, dataset, beta in datasets:
    if "dbpedia" in name:
        wordnet = wn.Wordnet("dbpedia14-en")
    elif "imagenet" in name:
        wordnet = wn.Wordnet("omw-subset200-en")
    elif "gtzan" in name:
        wordnet = wn.Wordnet("gtzan-en")
    else:
        raise ValueError(f"No mapping to wordnet for {name}")
    print(f"Dataset: {name}")
    leaves = algo.utils.all_leaves(wordnet)
    df = pd.read_csv(dataset)
    df['covered'] = False
    depths = pd.Series(dtype=object)
    best_cover_lens = []
    
    df['prediction_set_ss'] = df['prediction_set'].apply
    
    df = pd.read_csv(dataset)
    df['covered'] = False
    df['prediction_set_ss'] = None
    depths = pd.Series(dtype=object)
    best_cover_lens = []
    df['prediction_set_ss'] = df['prediction_set'].map(
        lambda x: frozenset(map(to_synset, x[1:-1].split(", ")))
    )
    df['label_ss'] = df['label'].map(to_synset)
    df['covered'] = df.apply(
        lambda x: algo.no_cover.covers_all_nodes(x['prediction_set_ss'], frozenset({x['label_ss']})),
        axis=1
    )
    df['n_covered'] = df.apply(
        lambda x: len(algo.no_cover.covered_nodes(x['prediction_set_ss'], leaves)),
        axis=1
    )
    df['costs'] = df['prediction_set_ss'].map(
        lambda x:
            algo.costs.cost(
                beta,
                x,
                leaves,
                None,
                cost_function_empty = 0.0
        )[0]
    )

Dataset: dbpedia


In [ ]:
df.groupby('beta')['covered'].mean()

beta
0.0    0.999893
0.1    0.999875
0.2    0.999857
0.3    0.999804
0.4    0.999804
0.5    0.999804
0.6    0.999804
0.7    0.999786
0.8    0.999786
0.9    0.999768
1.0    0.999768
Name: covered, dtype: float64

In [ ]:
df.groupby('beta')['prediction_set_len'].mean()

beta
0.0    1.000000
0.1    1.087875
0.2    1.289411
0.3    1.421982
0.4    1.476589
0.5    1.594929
0.6    1.616429
0.7    1.703982
0.8    1.710196
0.9    1.755804
1.0    1.957482
Name: prediction_set_len, dtype: float64

In [ ]:
df.groupby('beta')['n_covered'].mean()

beta
0.0    6.319500
0.1    5.327786
0.2    4.113554
0.3    3.647125
0.4    3.486679
0.5    3.227482
0.6    3.191607
0.7    3.054750
0.8    3.046375
0.9    2.993161
1.0    2.791482
Name: n_covered, dtype: float64

In [ ]:

alphas = ["0.01", "0.02", "0.05", ".1", ".15", ".2", ".3"]
datasets = [
    ("gtzan hcc", f"data/gtzan/hcp_ALPHA_{beta_gtz}.csv", beta_gtz),
    ("gtzan cp", f"data/gtzan/hcp_ALPHA_leafs.csv", beta_gtz),
    ("dbpedia14 hcc", f"data/dbpedia14/hcp_ALPHA_{beta_dbp}.csv", beta_dbp),
    ("dbpedia14 cp", f"data/dbpedia14/hcp_ALPHA_leafs.csv", beta_dbp),
    ("imagenet hcc", f"data/imagenet/hcp_ALPHA_{beta_img}.csv", beta_img),
    ("imagenet cp", f"data/imagenet/hcp_ALPHA_leafs.csv", beta_img),
]
# results = {dataset[0]: {} for dataset in datasets}
# a for-loop over alphas that runs the analysis on each alpha and stores the result in a result dict
for alpha in alphas:
    for name, dataset_base, beta in datasets:
        dataset = dataset_base.replace("ALPHA", str(alpha))
        print(f"Dataset: {name} alpha: {alpha}")
        if "dbpedia" in name:
            wordnet = wn.Wordnet("dbpedia14-en")
        elif "imagenet" in name:
            wordnet = wn.Wordnet("omw-subset-en")
        elif "gtzan" in name:
            wordnet = wn.Wordnet("gtzan-en")
        else:
            raise ValueError(f"No mapping to wordnet for {name}")
        leaves = algo.utils.all_leaves(wordnet)
        df = pd.read_csv(dataset)
        df['covered'] = False
        depths = pd.Series(dtype=object)
        best_cover_lens = []
        df['prediction_set_ss'] = df['prediction_set'].map(
            lambda x: frozenset(map(to_synset, x[1:-1].split(", ")))
        )
        df['label_ss'] = df['label'].map(to_synset)
        df['covered'] = df.apply(
            lambda x: algo.no_cover.covers_all_nodes(x['prediction_set_ss'], frozenset({x['label_ss']})),
            axis=1
        )
        df['n_covered'] = df.apply(
            lambda x: len(algo.no_cover.covered_nodes(x['prediction_set_ss'], leaves)),
            axis=1
        )
        df['costs'] = df['prediction_set_ss'].map(
            lambda x:
                algo.costs.cost(
                    beta,
                    x,
                    leaves,
                    None,
                    cost_function_empty = 0.0
            )[0]
        )
        results[name][alpha] = {
            'covered': df.covered.mean(),
            'covered_std': df.covered.std(),
            'covered_sem': df.covered.sem(),
            'alpha': df.alpha.mean(),
            'bf_alpha': df.bf_alpha.mean(),
            'beta': df.beta.mean(),
            'ps_len_mean': df.prediction_set_len.mean(),
            'ps_lens': df.prediction_set_len,
            # 'label_depth': df.label_depth,
            # 'label_dists': depths - df.label_depth,
            # 'depths': depths,
            'n_covered': df.n_covered,
            'best_cover_lens': df['best_cover'].apply(len),
            'df': df,
            }

Dataset: gtzan hcc alpha: 0.01
Dataset: gtzan cp alpha: 0.01
Dataset: gtzan hcc alpha: 0.02


In [ ]:
# output mean and std for every alpha for this dataset. The output should be usable in a tikz plot
for name, _ , _ in datasets:
    print(f"Results for dataset: {name}")
    for alpha in alphas:
        result = results[name][alpha]
        print(f"Alpha: {alpha}, Coverage: {result['covered']:.4f} pm {result['covered_std']:.4f}, PS size: {result['ps_len_mean']:.2f} pm {result['ps_lens'].std():.2f}, # covered: {result['n_covered'].mean():.2f} pm {result['n_covered'].std():.2f}")

Results for dataset: gtzan hcc
Alpha: 0.01, Coverage: 1.0000 pm 0.0000, PS size: 1.05 pm 0.22, # covered: 9.48 pm 1.96
Alpha: 0.02, Coverage: 0.9925 pm 0.0863, PS size: 1.07 pm 0.26, # covered: 8.18 pm 3.51
Alpha: 0.05, Coverage: 0.9812 pm 0.1357, PS size: 1.14 pm 0.34, # covered: 7.34 pm 3.88
Alpha: .1, Coverage: 0.9613 pm 0.1931, PS size: 1.19 pm 0.39, # covered: 6.17 pm 4.14
Alpha: .15, Coverage: 0.9363 pm 0.2445, PS size: 1.23 pm 0.42, # covered: 5.29 pm 4.14
Alpha: .2, Coverage: 0.8762 pm 0.3295, PS size: 1.22 pm 0.42, # covered: 4.21 pm 3.95
Alpha: .3, Coverage: 0.6800 pm 0.4668, PS size: 1.19 pm 0.39, # covered: 1.37 pm 0.95
Results for dataset: gtzan cp
Alpha: 0.01, Coverage: 1.0000 pm 0.0000, PS size: 1.05 pm 0.22, # covered: 9.48 pm 1.96
Alpha: 0.02, Coverage: 0.9925 pm 0.0863, PS size: 1.07 pm 0.26, # covered: 8.18 pm 3.51
Alpha: 0.05, Coverage: 0.9812 pm 0.1357, PS size: 1.14 pm 0.34, # covered: 7.34 pm 3.88
Alpha: .1, Coverage: 0.9613 pm 0.1931, PS size: 1.19 pm 0.39, # co